In [1]:
!apt-get update -qq
!apt-get install -y flex bison gcc

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
gcc is already the newest version (4:11.2.0-1ubuntu1).
gcc set to manually installed.
The following additional packages will be installed:
  libfl-dev libfl2
Suggested packages:
  bison-doc flex-doc
The following NEW packages will be installed:
  bison flex libfl-dev libfl2
0 upgraded, 4 newly installed, 0 to remove and 78 not upgraded.
Need to get 1,072 kB of archives.
After this operation, 3,667 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 flex amd64 2.6.4-8build2 [307 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 bison amd64 2:3.8.2+dfsg-1build1 [748 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 libfl2 amd64 2.6.4-8build2 

In [2]:
%%writefile tac.l
%{
#include "tac.tab.h"
#include <string.h>
#include <stdlib.h>
%}

%%

[a-zA-Z][a-zA-Z0-9]* {
    yylval.str = strdup(yytext);
    return ID;
}

[0-9]+ {
    yylval.str = strdup(yytext);
    return NUM;
}

[\t\n ]+ {
    /* skip spaces */
}

. {
    return yytext[0];
}

%%

int yywrap()
{
    return 1;
}

Writing tac.l


In [3]:
%%writefile tac.y
%{
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

int tempCount = 1;
char temp[10];

int yylex(void);
int yyerror(char *s);
%}

%union {
    char *str;
}

%token <str> ID NUM

%type <str> expr

%left '+' '-'
%left '*' '/'

%%

stmt:
    ID '=' expr
    {
        printf("%s = %s\n", $1, $3);
    }
    ;

expr:
    expr '+' expr
    {
        sprintf(temp, "t%d", tempCount++);
        printf("%s = %s + %s\n", temp, $1, $3);
        $$ = strdup(temp);
    }

    | expr '-' expr
    {
        sprintf(temp, "t%d", tempCount++);
        printf("%s = %s - %s\n", temp, $1, $3);
        $$ = strdup(temp);
    }

    | expr '*' expr
    {
        sprintf(temp, "t%d", tempCount++);
        printf("%s = %s * %s\n", temp, $1, $3);
        $$ = strdup(temp);
    }

    | expr '/' expr
    {
        sprintf(temp, "t%d", tempCount++);
        printf("%s = %s / %s\n", temp, $1, $3);
        $$ = strdup(temp);
    }

    | ID
    {
        $$ = $1;
    }

    | NUM
    {
        $$ = $1;
    }
    ;

%%

int main()
{
    printf("Enter the expression:\n");
    yyparse();
    return 0;
}

int yyerror(char *s)
{
    printf("Error: %s\n", s);
    return 0;
}

Writing tac.y


In [4]:
!bison -d tac.y

In [5]:
!flex tac.l

In [6]:
!gcc lex.yy.c tac.tab.c -o tac -lfl

In [7]:
!echo "a=b+c" | ./tac


Enter the expression:
t1 = b + c
a = t1


In [8]:
!echo "a=b+c*d" | ./tac

Enter the expression:
t1 = c * d
t2 = b + t1
a = t2


In [9]:
!echo "x=a+b*c-d" | ./tac

Enter the expression:
t1 = b * c
t2 = a + t1
t3 = t2 - d
x = t3
